# Add Bedrock Guardrails

This is the "why Gateway" notebook. Up to now, Path B has been functionally similar to Path A for Docker-based tools -- just with audit logging and JWT auth. In this notebook, you will add **content guardrails** that screen every tool response for sensitive information before it reaches the agent.

This is something Path A (NGINX) fundamentally cannot do. NGINX is a reverse proxy -- it forwards bytes without inspecting content. The AgentCore Gateway's response interceptor Lambda, on the other hand, can call Amazon Bedrock Guardrails to block PII, harmful content, or anything else that violates your enterprise policy.

**What you will do:**
1. Create a Bedrock Guardrail that blocks PII (Social Security numbers, credit card numbers) and harmful content
2. Update the response interceptor Lambda with the guardrail ID
3. Test Path B with content that triggers the guardrail
4. Show that the same content passes through Path A unfiltered

## Step 1: Create a Bedrock Guardrail

We use the Bedrock API to create a guardrail with two policies:
- **Content filter** -- blocks harmful content (hate, violence, sexual, misconduct)
- **Sensitive information filter** -- blocks PII like SSNs, credit card numbers, and email addresses

The guardrail will be applied by the response interceptor Lambda to every tool response that passes through Path B.

In [ ]:
import boto3, json, time

region = boto3.session.Session().region_name or "us-west-2"
bedrock = boto3.client("bedrock", region_name=region)

GUARDRAIL_NAME = "workshop-tool-guardrail"

# Check if guardrail already exists (idempotent)
_existing = []
_gr_paginator = bedrock.get_paginator("list_guardrails")
for _page in _gr_paginator.paginate():
    _existing.extend(_page.get("guardrails", []))
existing_gr = [g for g in _existing if g["name"] == GUARDRAIL_NAME]

if existing_gr:
    GUARDRAIL_ID = existing_gr[0]["id"]
    GUARDRAIL_VERSION = existing_gr[0].get("version", "DRAFT")
    print(f"Guardrail already exists: {GUARDRAIL_ID} (version: {GUARDRAIL_VERSION})")
else:
    # Create the guardrail
    response = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        description="Enterprise guardrail for tool outputs: blocks PII and harmful content",
        blockedInputMessaging="This request has been blocked by enterprise guardrails.",
        blockedOutputsMessaging="This tool response has been blocked because it contains sensitive information.",
        # Content policy: block harmful content
        contentPolicyConfig={
            "filtersConfig": [
                {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            ]
        },
        # Sensitive information policy: block PII
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
                {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "PHONE", "action": "ANONYMIZE"},
                {"type": "US_BANK_ACCOUNT_NUMBER", "action": "BLOCK"},
            ]
        },
    )

    GUARDRAIL_ID = response["guardrailId"]
    GUARDRAIL_VERSION = response["version"]
    print(f"Created guardrail: {GUARDRAIL_ID}")
    print(f"  Version: {GUARDRAIL_VERSION}")

print(f"\nGuardrail ID:      {GUARDRAIL_ID}")
print(f"Guardrail Version: {GUARDRAIL_VERSION}")
print(f"\nThis guardrail will:")
print(f"  - BLOCK responses containing SSNs, credit card numbers, bank accounts")
print(f"  - ANONYMIZE email addresses and phone numbers")
print(f"  - BLOCK harmful content (hate, violence, sexual, misconduct, insults)")

## Step 2: Update the Response Interceptor Lambda

The response interceptor Lambda needs to know the guardrail ID and version so it can call `bedrock-runtime.apply_guardrail()` on every tool response. We update the Lambda's environment variables using a **read-merge-update** pattern to avoid overwriting any existing configuration.

In [ ]:
lambda_client = boto3.client("lambda", region_name=region)

RESPONSE_INTERCEPTOR_FN = "agentcore-gateway-response-interceptor"

# Read current configuration (read-merge-update pattern)
current_config = lambda_client.get_function_configuration(FunctionName=RESPONSE_INTERCEPTOR_FN)
current_env = current_config.get("Environment", {}).get("Variables", {})

print("Current environment variables:")
for k, v in current_env.items():
    print(f"  {k} = {v}")

# Merge in the guardrail configuration
# The interceptor Lambda reads BEDROCK_GUARDRAIL_ID and BEDROCK_GUARDRAIL_VERSION
updated_env = {**current_env}
updated_env["BEDROCK_GUARDRAIL_ID"] = GUARDRAIL_ID
updated_env["BEDROCK_GUARDRAIL_VERSION"] = GUARDRAIL_VERSION

# Update the Lambda
lambda_client.update_function_configuration(
    FunctionName=RESPONSE_INTERCEPTOR_FN,
    Environment={"Variables": updated_env},
)

print(f"\nUpdated {RESPONSE_INTERCEPTOR_FN} environment:")
for k, v in updated_env.items():
    print(f"  {k} = {v}")

# Wait for the update to propagate
print("\nWaiting for Lambda update to propagate...")
waiter = lambda_client.get_waiter("function_updated_v2")
waiter.wait(FunctionName=RESPONSE_INTERCEPTOR_FN, WaiterConfig={"Delay": 2, "MaxAttempts": 30})
print("Lambda updated and ready.")

## Step 3: Test Path B -- Trigger the Guardrail

Now we will send a request through Path B where the tool response contains a fake Social Security number. The response interceptor should detect the SSN via Bedrock Guardrails and block or redact it.

We simulate this by calling a tool and expecting the response interceptor to screen the output. If the tool itself returns PII, the guardrail will catch it.

**Note:** Since we cannot control the actual tool output in this test, we will use the Bedrock Runtime `apply_guardrail` API directly to demonstrate the guardrail behavior, then show the end-to-end flow through the gateway.

In [ ]:
bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

# Simulate a tool response that contains PII.
# Synthetic test data — standard industry placeholders only (000-00-0000 SSN,
# 4111-1111-1111-1111 test card, example.com, 555 phone). No real PII is used.
test_content_with_pii = (
    "Customer record found. Name: John Smith, "
    "SSN: 000-00-0000, "
    "Credit Card: 4111-1111-1111-1111, "
    "Email: john.smith@example.com, "
    "Phone: (555) 123-4567. "
    "Account balance: $12,450.00."
)

print("Testing Bedrock Guardrail with simulated tool output containing PII")
print("=" * 70)
print(f"\nOriginal tool output:")
print(f"  {test_content_with_pii}")
print()

# Apply the guardrail to the content (this is what the response interceptor does)
guardrail_response = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION,
    source="OUTPUT",  # We are screening tool OUTPUT, not user input
    content=[{"text": {"text": test_content_with_pii}}],
)

action = guardrail_response.get("action", "NONE")
outputs = guardrail_response.get("outputs", [])
assessments = guardrail_response.get("assessments", [])

print(f"Guardrail action: {action}")
print()

if action == "GUARDRAIL_INTERVENED":
    print("The guardrail BLOCKED or MODIFIED the response.")
    print(f"\nFiltered output:")
    for output in outputs:
        print(f"  {output.get('text', '?')}")
else:
    raise AssertionError(
        f"Expected GUARDRAIL_INTERVENED for content with SSN/credit-card/email/phone, "
        f"got action={action}. The guardrail is not configured correctly — "
        f"verify sensitiveInformationPolicyConfig in cell 2."
    )

# Show what the guardrail detected
print(f"\nAssessment details:")
for assessment in assessments:
    sensitive = assessment.get("sensitiveInformationPolicy", {})
    if sensitive:
        pii_findings = sensitive.get("piiEntities", [])
        for finding in pii_findings:
            print(f"  PII detected: {finding.get('type', '?')} -> action: {finding.get('action', '?')}")
    content_policy = assessment.get("contentPolicy", {})
    if content_policy:
        filters = content_policy.get("filters", [])
        for f in filters:
            print(f"  Content filter: {f.get('type', '?')} -> confidence: {f.get('confidence', '?')}")

## Step 4: Architectural Contrast -- Content Screening

The Step 3 `apply_guardrail` call above is the exact code path the Path B response interceptor runs on every tool response. Path A (NGINX) has no equivalent hook — it is a byte-forwarding reverse proxy.

This cell is **not** a live test against Path A; no such endpoint exists to hit. It reprints the same synthetic `test_content_with_pii` string twice and labels how each path would handle it, as an architectural summary. Treat it as a diagram, not a measurement.

In [ ]:
# Path A vs Path B comparison for the same content

print("Side-by-Side Comparison: Content Screening")
print("=" * 70)

print("\nOriginal tool output (contains SSN, credit card, email, phone):")
print(f"  {test_content_with_pii}")

print("\n--- PATH A (NGINX) ---")
print("  Result: PASSED THROUGH UNFILTERED")
print(f"  Agent receives: {test_content_with_pii}")
print("  PII detected: None (NGINX does not inspect content)")

print("\n--- PATH B (AgentCore Gateway + Bedrock Guardrails) ---")
print(f"  Guardrail action: {action}")
if outputs:
    filtered_text = outputs[0].get("text", "?")
    print(f"  Agent receives: {filtered_text}")
else:
    print("  Agent receives: (blocked message)")

print()
print("Key takeaway:")
print("  Path A is faster but blind to content. Path B adds a content")
print("  screening layer that enforces enterprise data protection policies.")
print("  For regulated industries (healthcare, finance, government), this is")
print("  not optional -- it is a compliance requirement.")

## How Guardrails Work in the Response Interceptor

The response interceptor Lambda is invoked by the AgentCore Gateway after every tool response. Here is the flow:

```
Tool Target returns response
       |
       v
Response Interceptor Lambda
  1. Parse the tool response body
  2. If GUARDRAIL_ID is set in env vars:
     a. Call bedrock-runtime.apply_guardrail(source="OUTPUT", content=response_text)
     b. If action == "GUARDRAIL_INTERVENED":
        - Replace response body with the filtered/blocked message
        - Log the guardrail action
     c. If action == "NONE":
        - Pass response through unchanged
  3. Return the (possibly modified) response to the gateway
       |
       v
Gateway returns response to the agent
```

The guardrail evaluation adds latency (typically 100-300ms), but it happens in the Lambda -- not in the critical path of the agent's reasoning. The tradeoff is worthwhile for workloads that handle sensitive data.

**Next:** In notebook 07, you will register the AgentCore Gateway itself as a discoverable service in the Registry -- completing the composability loop.